---
# 🚇 Urban Growth Dynamics of ASEAN Megacities
## Through the Lens of Public Transport Ridership
### การวิเคราะห์การเติบโตของมหานครอาเซียนผ่านเลนส์ระบบขนส่งสาธารณะ
---
## 📋 Executive Summary — บทสรุปผู้บริหาร

&emsp;ลองนึกภาพตามนะคะ — ทุกเช้า คนหลายสิบล้านคนในอาเซียนออกจากบ้าน บางคนขึ้น BTS บางคนนั่ง MRT สิงคโปร์ บางคนต่อรถเมล์ในกัวลาลัมเปอร์ การเคลื่อนไหวของคนเหล่านี้ไม่ใช่แค่ "การเดินทาง" — มันคือ **สัญญาณชีพ** ของเมือง

&emsp;เมื่อเศรษฐกิจดี คนออกไปทำงาน ช้อปปิ้ง พบปะสังสรรค์ — ตัวเลขผู้โดยสารพุ่งขึ้น เมื่อเกิดวิกฤต เช่น COVID-19 — ตัวเลขดิ่งลงในชั่วข้ามคืน นี่คือสิ่งที่รายงานฉบับนี้ต้องการพิสูจน์

| ส่วน | เมือง/หัวข้อ | สิ่งที่จะค้นพบ |
|------|-------------|----------------|
| **Part 1** | กรุงเทพฯ 🇹🇭 | คนกรุงเทพฯ เดินทางด้วยอะไร เพิ่มขึ้นแค่ไหน? |
| **Part 2** | สิงคโปร์ 🇸🇬 | ระบบขนส่งระดับโลกเขาทำได้อย่างไร? |
| **Part 3** | กัวลาลัมเปอร์ 🇲🇾 | มาเลย์กำลังไปในทิศทางไหน? |
| **Part 4** | จาการ์ตา 🇮🇩 | เมืองที่ใหญ่ที่สุดในอาเซียนอยู่ตรงไหน? |
| **Part 5** | มะนิลา 🇵🇭 | ฟิลิปปินส์โตมา 25 ปีอย่างไร? |
| **Part 6** | Economic Growth | ขนส่งกับเศรษฐกิจ — มีความสัมพันธ์กันจริงไหม? |
| **Part 7** | Development Gap | เมืองกำลังพัฒนาตามสิงคโปร์ทัน? |

---

## 🗂️ ที่มาและความสำคัญ

&emsp;อาเซียนมีประชากรกว่า **650 ล้านคน** กระจายอยู่ใน 10 ประเทศ และกำลังก้าวเข้าสู่ยุคของการขยายตัวอย่างรวดเร็ว เมืองใหญ่ๆ เช่น กรุงเทพฯ จาการ์ตา และมะนิลา ต่างแบกรับประชากรหลักสิบล้านคน พร้อมกับความท้าทายด้านการจราจรและมลพิษที่ตามมา

&emsp;ระบบขนส่งสาธารณะจึงไม่ใช่แค่ "รถไฟฟ้า" หรือ "รถเมล์" — มันคือตัวชี้วัดว่าเมืองนั้น **พัฒนาไปถึงไหน** แล้ว

**ข้อมูลที่ใช้วิเคราะห์:**
- 📊 จำนวนผู้โดยสารรายเดือน จำแนกตามระบบขนส่ง (2019–2025)
- 💰 GDP รายเมือง (Billion USD)
- 👥 จำนวนประชากร (ล้านคน)
- 🌫️ ค่าฝุ่น PM2.5 (μg/m³)

---

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Dark Theme ────────────────────────────────────────────────────────────────
PAPER_BG   = '#0D1117'
PLOT_BG    = '#161B22'
GRID_C     = '#30363D'
FONT_C     = '#E6EDF3'
MUTED_C    = '#8B949E'
TEMPLATE   = 'plotly_dark'

CITY_COLORS = {
    'Bangkok':      '#FF6B6B',
    'Singapore':    '#74B9FF',
    'Kuala Lumpur': '#4ECDC4',
    'Jakarta':      '#FFB347',
    'Manila':       '#C77DFF',
}

def base_layout(height=460, legend_h=True):
    leg = dict(bgcolor='rgba(22,27,34,0.9)', bordercolor=GRID_C, borderwidth=1, font_size=11)
    if legend_h:
        leg.update(orientation='h', y=1.13, x=1, xanchor='right')
    return dict(
        template=TEMPLATE, paper_bgcolor=PAPER_BG, plot_bgcolor=PLOT_BG,
        font=dict(color=FONT_C, family='Segoe UI, Arial, sans-serif'),
        height=height, legend=leg,
        margin=dict(l=60, r=40, t=75, b=55),
        hoverlabel=dict(bgcolor='#1F2937', bordercolor=GRID_C, font_size=12),
    )

def ax_style():
    return dict(gridcolor=GRID_C, zerolinecolor=GRID_C)

print('✅ Setup complete — Dark theme loaded')

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('ASEAN_Urban_Growth_Final_with_Mode.csv')
df['Date']  = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# ── Mode label maps ───────────────────────────────────────────────────────────
MODE_BKK = {'BTS':'BTS Skytrain','MRT':'MRT Blue/Purple','SRT':'SRT Red Line',
             'ARL':'Airport Rail Link','YL':'MRT Yellow Line','PK':'MRT Pink Line','RL':'Regional Rail'}
MODE_SGP = {'MRT':'MRT','Public Bus':'Public Bus','LRT':'LRT'}
MODE_KL  = {
    'rail_lrt_kj':'LRT Kelana Jaya','rail_mrt_kajang':'MRT Kajang',
    'rail_lrt_ampang':'LRT Ampang','bus_rkl':'RapidKL Bus',
    'rail_mrt_pjy':'MRT Putrajaya','rail_monorail':'KL Monorail',
    'rail_komuter':'KTM Komuter','rail_komuter_utara':'KTM Utara',
    'rail_ets':'ETS Train','rail_intercity':'Intercity Rail',
    'rail_tebrau':'KTM Tebrau','bus_rkn':'RapidKN Bus','bus_rpn':'RapidPN Bus'
}
MODE_JKT = {'TRANSJAKARTA':'TransJakarta (BRT)','KRL':'KRL Commuter',
             'MRT':'MRT Jakarta','LRT':'LRT Jakarta',
             'BUS SEKOLAH':'School Bus','KCI COMMUTER BANDARA':'Airport Rail','KAPAL':'Ferry'}

print(f'Dataset: {df.shape[0]:,} rows | {df["City"].nunique()} cities')
print('Cities:', df['City'].unique().tolist())

---
## Part 7 — Development Gap 🌏
### เมืองกำลังพัฒนาตามสิงคโปร์ทัน?

&emsp;สิงคโปร์มีประชากรเพียง 5.9 ล้านคน แต่กลับมีผู้โดยสารระบบขนส่งปีละกว่า **2,700 ล้านเที่ยว** คนสิงคโปร์ 1 คน ใช้ระบบขนส่งสาธารณะเฉลี่ยกว่า **450 เที่ยวต่อปี**

&emsp;เปรียบเทียบกับกรุงเทพฯ ที่มีคน 10 ล้านคน แต่มีผู้โดยสารเพียง ~540 ล้านเที่ยวต่อปี หรือประมาณ **50 เที่ยวต่อคนต่อปี** — ห่างกัน 9 เท่า!

&emsp;คำถามคือ ช่องว่างนี้กำลังแคบลงหรือไม่? ส่วนนี้วิเคราะห์ผ่าน 3 กราฟ: CAGR เปรียบเทียบ, Passenger per Capita และ Growth Index

---

In [ ]:
# ── Chart 7.1: CAGR Bar ───────────────────────────────────────────────────────
def calc_cagr(s, y0, y1):
    p = s.set_index('Year')['Total_Ridership']
    if y0 in p.index and y1 in p.index and p[y0]>0 and p[y1]>0:
        return ((p[y1]/p[y0])**(1/(y1-y0))-1)*100
    return np.nan

cagr_data = []
for city in yearly['City'].unique():
    v = calc_cagr(yearly[yearly['City']==city], 2021, 2024)
    if not np.isnan(v): cagr_data.append({'City':city,'CAGR':v})
cagr_df = pd.DataFrame(cagr_data).sort_values('CAGR')

sg_cagr = cagr_df[cagr_df['City']=='Singapore']['CAGR'].values[0]

fig = px.bar(cagr_df, x='CAGR', y='City', orientation='h',
    title='<b>ตารางที่ 7.1</b>  CAGR ผู้โดยสาร 2021–2024 (%)  |  ★ สิงคโปร์ = Developed Benchmark',
    labels={'CAGR':'CAGR (%)','City':''},
    color='City', color_discrete_map=CITY_COLORS, text_auto='.1f')
fig.add_vline(x=sg_cagr, line_dash='dot', line_color=CITY_COLORS['Singapore'],
              annotation_text='Singapore benchmark',
              annotation_position='top', annotation_font_color=CITY_COLORS['Singapore'],
              annotation_font_size=10)
fig.update_layout(**base_layout(380, legend_h=False), showlegend=False,
                  xaxis=dict(ticksuffix='%',**ax_style()), yaxis=ax_style())
fig.update_traces(textfont_size=11, textposition='outside')
fig.show()

# ── Chart 7.2: Passenger per Capita ──────────────────────────────────────────
ppc = df.groupby(['Year','City'])['Passenger_per_Capita'].sum().reset_index()
ppc = ppc[ppc['Year'].between(2019,2025)]
ppc = ppc[~((ppc['Year']==2025)&(ppc['City'].isin(['Kuala Lumpur','Manila'])))]

fig = px.line(ppc, x='Year', y='Passenger_per_Capita', color='City',
    markers=True, line_shape='spline',
    title='<b>ตารางที่ 7.2</b>  เที่ยวโดยสารต่อคนต่อปี (Passenger per Capita) — วัดระดับการพึ่งพา',
    labels={'Passenger_per_Capita':'เที่ยวต่อคนต่อปี','Year':''},
    color_discrete_map=CITY_COLORS)
fig.update_layout(**base_layout(480), hovermode='x unified',
                  xaxis=dict(dtick=1,**ax_style()), yaxis=ax_style())
fig.update_traces(line_width=2.5, marker_size=7)
fig.show()

# ── Chart 7.3: Growth Index (2021 = 100) ─────────────────────────────────────
base_yr = yearly[yearly['Year'].between(2021,2024)].copy()
idx_list=[]
for city in base_yr['City'].unique():
    sub = base_yr[base_yr['City']==city].sort_values('Year').copy()
    if 2021 in sub['Year'].values:
        base_val = sub[sub['Year']==2021]['Total_Ridership'].values[0]
        sub['Index'] = sub['Total_Ridership']/base_val*100
        idx_list.append(sub)
idx_df = pd.concat(idx_list)

fig = px.line(idx_df, x='Year', y='Index', color='City',
    markers=True, line_shape='spline',
    title='<b>ตารางที่ 7.3</b>  Growth Index (ปี 2021 = 100) — เปรียบเทียบอัตราเติบโตอย่างเป็นธรรม',
    labels={'Index':'Growth Index (2021=100)','Year':''},
    color_discrete_map=CITY_COLORS)
fig.add_hline(y=100, line_dash='dash', line_color=MUTED_C, line_width=1.2)
fig.update_layout(**base_layout(480), hovermode='x unified',
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(ticksuffix=' idx',**ax_style()))
fig.update_traces(line_width=2.5, marker_size=7)
fig.show()

> 📌 **สรุป Part 7 — Development Gap:**
>
> ตอบคำถามตรงๆ ก่อนเลย — **ช่องว่างกำลังแคบลง แต่ยังห่างกันมาก**
>
> ใน CAGR Chart เห็นว่า Bangkok มีอัตราเติบโตสูงกว่า Singapore อย่างชัดเจน ซึ่งฟังดูดี แต่ต้องทำความเข้าใจว่า Singapore เติบโตช้าไม่ใช่เพราะระบบแย่ลง — แต่เพราะ **ระบบอิ่มตัวแล้ว** คนสิงคโปร์ใช้ระบบขนส่งเกือบเต็มศักยภาพอยู่แล้ว จึงไม่มีพื้นที่ให้โตมาก
>
> ในทางกลับกัน Bangkok, KL, Manila ยังมีคนจำนวนมากที่ยังพึ่งพารถยนต์ส่วนตัว ทุก Rail สายใหม่ที่เปิด คือโอกาสในการเปลี่ยนพฤติกรรมของคนกลุ่มนั้น
>
> **ถ้าอัตราเติบโตนี้รักษาไว้ได้** — ใน 10–15 ปีข้างหน้า เราอาจเห็น Bangkok หรือ KL มีระบบขนส่งที่เทียบเคียงสิงคโปร์ได้ในแง่ความหนาแน่นการใช้งาน

---

## 📝 บทสรุปการศึกษา — สรุปให้เข้าใจง่าย

### ภาพรวมที่เห็น

&emsp;ถ้าจะสรุปทั้งหมดนี้ในประโยคเดียว คือ: **"อาเซียนกำลังเดินทางสู่เมืองที่ดีกว่า แต่ยังไม่ถึงจุดหมาย"**

&emsp;ทุกเมืองในกลุ่มนี้กำลังลงทุนในระบบขนส่ง ทุกเมืองมีตัวเลขผู้โดยสารที่เพิ่มขึ้น และทุกเมืองมีความต้องการการเดินทางที่ยังโตได้อีกมาก แต่แต่ละเมืองมีสถานการณ์และความท้าทายต่างกัน

### สรุปรายเมือง

| เมือง | สถานการณ์ปัจจุบัน | จุดแข็ง | ความท้าทาย |
|-------|------------------|---------|------------|
| 🇹🇭 **กรุงเทพฯ** | กำลังฟื้นตัวและขยายตัวอย่างรวดเร็ว | Rail หลายสายใหม่กำลังได้รับความนิยม | การเชื่อมต่อระหว่างสายและ Last Mile ยังเป็นปัญหา |
| 🇸🇬 **สิงคโปร์** | ระบบอิ่มตัวและมีคุณภาพสูงมาก | Bus + MRT ทำงานเป็นเครือข่ายเดียวกัน | ไม่มีพื้นที่ขยายมาก ต้องรักษาคุณภาพ |
| 🇲🇾 **KL** | ฟื้นตัวแล้วและกำลังเติบโต | ระบบ Rail กำลังขยาย | ประชากรยังพึ่งรถยนต์มากเกินไป |
| 🇮🇩 **จาการ์ตา** | เพิ่งเริ่มพัฒนาจริงจัง | TransJakarta ใหญ่มาก, MRT เริ่มแล้ว | PM2.5 สูงที่สุด ต้องการการพัฒนาเร่งด่วน |
| 🇵🇭 **มะนิลา** | ระบบ Rail ไม่เพียงพอกับความต้องการ | ข้อมูลยาวที่สุด 25 ปี | ขีดความสามารถระบบเดิมเต็มแล้ว ต้องสร้างใหม่ |

### คำตอบของ 2 คำถามหลัก

**① ขนส่งสาธารณะสะท้อนเศรษฐกิจไหม?**

&emsp;✅ **ใช่ — อย่างชัดเจน** ทุกเมืองแสดงให้เห็นว่าเมื่อเศรษฐกิจดี คนออกเดินทางมาก เมื่อวิกฤต คนหยุดเดินทาง ความสัมพันธ์นี้ชัดเจนจนนักเศรษฐศาสตร์บางกลุ่มเสนอให้ใช้ตัวเลข Ridership เป็น Real-time Economic Indicator แทนการรอ GDP ที่ออกช้ากว่า

**② เมืองกำลังพัฒนาตามสิงคโปร์ทัน?**

&emsp;⚡ **ยังไม่ทัน แต่กำลังวิ่ง** ช่องว่างในแง่ตัวเลขสัมบูรณ์ยังใหญ่มาก แต่ในแง่อัตราเติบโต Bangkok, KL และ Jakarta แสดงการเร่งตัวที่น่าประทับใจ ถ้าการลงทุนในระบบขนส่งยังคงต่อเนื่อง และนโยบายสนับสนุนการใช้ระบบขนส่งสาธารณะแทนรถยนต์ส่วนตัวได้ผล — **ทศวรรษหน้าจะน่าจับตามากที่สุด**

---
*Data: ASEAN_Urban_Growth_Final_with_Mode.csv | Framework: Urban Growth Dynamics of ASEAN Megacities*